# AeroVision Lab — saved-model evaluation
This notebook does **not train**. It evaluates the saved `colab-pilot-v1` model
at prediction limits 300, 600 and 1000 using the same 200 validation images.
Select a hosted GPU runtime, mount the same Google Drive, then run all cells.
Keep private: source code is embedded. No GitHub token is needed.


In [ ]:
import os, sys, pathlib, subprocess, json, csv, zipfile, io, base64, hashlib
from google.colab import drive
import torch
assert torch.cuda.is_available(), 'Select a hosted GPU runtime, then rerun this cell.'
print('Hosted GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PERSIST = pathlib.Path('/content/drive/MyDrive/AeroVisionLab')
PERSIST.mkdir(parents=True, exist_ok=True)
PROJECT = pathlib.Path('/content/aerovision-lab')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT / '.state/ultralytics')


Source snapshot SHA-256: `6389d0a7a815dc8aa9864ce988b0d5261cd2ee43dbea8ade2a2ad05b0eca1a43`. The next cell installs this snapshot into the hosted runtime.

In [ ]:
bundle = base64.b64decode('...')
assert hashlib.sha256(bundle).hexdigest() == '6389d0a7a815dc8aa9864ce988b0d5261cd2ee43dbea8ade2a2ad05b0eca1a43'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert (PROJECT / member).resolve().is_relative_to(PROJECT)
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
# Editable-install path hooks are not reloaded by an already-running kernel.
sys.path.insert(0, str(PROJECT / 'src'))
# Save the source that produced the run alongside its outputs.
(PERSIST / 'source-6389d0a7a815.zip').write_bytes(bundle)
def run(*args):
    command = [sys.executable, '-u', '-m', 'aerovision.cli', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
run('doctor')


In [ ]:
# Verify the model before preparing data or doing inference.
from aerovision.data import digest
BEST = PERSIST / 'runs/colab-pilot-v1/weights/best.pt'
assert BEST.exists(), f'Checkpoint not found: {BEST}'
EXPECTED_MODEL = '3ecd854c6b6607edabc455128057b3a77a2816898a84ea405c178fd08e307194'
assert digest(BEST) == EXPECTED_MODEL, 'This is not the checkpoint from the pilot reviewed in chat.'
saved_manifest = PERSIST / 'manifest.json'
assert saved_manifest.exists(), 'Original subset manifest is missing from Drive.'
original = json.loads(saved_manifest.read_text())
run('prepare', '--train', 800, '--val', 200)
DATA = PROJECT / 'data/mini-t800-v200-s42/dataset.yaml'
subset = DATA.parent
current = json.loads((subset / 'manifest.json').read_text())
assert original['splits'] == current['splits'], 'Image membership or content differs from the pilot.'
(PERSIST / 'runs').mkdir(exist_ok=True)
if not pathlib.Path('runs').exists():
    pathlib.Path('runs').symlink_to(PERSIST / 'runs', target_is_directory=True)
assert pathlib.Path('runs').resolve() == (PERSIST / 'runs').resolve()


## Controlled prediction-limit sweep
The COCO evaluator limit stays at 1000 per category for every run. The prediction
limit varies jointly at the detector output (per tile for tiled mode) and after
global merging. This does not isolate those two stages from each other.
The 300 control uses the original prediction limits, but a different evaluator
limit from the original report. Compare within this new sweep.


In [ ]:
import pandas as pd
rows = []
DESTINATIONS = []
for cap in [300, 600, 1000]:
    destination = pathlib.Path('runs') / f'colab-pilot-v1-cap-{cap}'
    print(f'\nEvaluating prediction cap {cap}', flush=True)
    run('compare', '--data', DATA, '--weights', BEST, '--max-det', cap,
        '--eval-max-det', 1000, '--output', destination, '--device', '0')
    report = json.loads((destination / 'comparison.json').read_text())
    for mode, values in report['modes'].items():
        rows.append(dict(cap=cap, mode=mode, AP=values['ap'], AP50=values['ap50'],
            AP_small=values['ap_small'], AR_small=values['recall_small'],
            p50_ms=values['latency_p50_ms'], p95_ms=values['latency_p95_ms'],
            capped_images=values['images_at_prediction_cap']))
    DESTINATIONS.append(destination)
table = pd.DataFrame(rows)
display(table)
SWEEP = PERSIST / 'runs/colab-pilot-v1-cap-sweep'
SWEEP.mkdir(exist_ok=True)
table.to_csv(SWEEP / 'summary.csv', index=False)


In [ ]:
# Fixed confidence diagnostic, separate from COCO AP.
from aerovision.diagnostics import review
from IPython.display import display, Image as DisplayImage
for destination in DESTINATIONS:
    errors = review(subset, destination, confidence=0.25, examples=6)
    print(destination.name, {m: v['totals'] for m, v in errors['modes'].items()})
# Same first six validation images across modes; these are not cherry-picked successes.
for path in sorted((DESTINATIONS[-1] / 'error-review').glob('*.jpg'))[:4]:
    display(DisplayImage(filename=str(path), width=1000))


In [ ]:
# Save metrics and visual examples, excluding bulky prediction JSON and model duplicates.
export = PERSIST / 'colab-pilot-v1-evaluation-v2.zip'
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
    for directory in [SWEEP, *DESTINATIONS]:
        for path in directory.rglob('*'):
            if path.is_file() and not path.name.endswith('.predictions.json'):
                archive.write(path, path.resolve().relative_to((PERSIST / 'runs').resolve()))
print('Saved to Drive:', export)
from google.colab import files
files.download(str(export))


Send back `colab-pilot-v1-evaluation-v2.zip`. This is a validation diagnostic,
not a final held-out test or evidence that a higher cap is better. No local GPU is used.
